### protein conc vs salt concentrations'

In [1]:
import json
import math
import time
import toml
#import matplotlib.pyplot as plt
import numpy as np
import opentrons
import pandas as pd
#from matplotlib.colors import ListedColormap
from opentrons import execute, simulate
from opentrons.types import Location, Point
import datetime

In [2]:
class Fluid:
    def __init__(
        self,
        name,
        location,
        salt_concentration: float,
        pH: float,
        transfers: list,
        protein_concentration=0,
    ):
        self.name = name
        if not isinstance(location, opentrons.protocol_api.labware.Well):
            raise TypeError(
                "Location must be of type opentrons.protocol_api.labware.Well"
            )
        self.location = location

        if not isinstance(salt_concentration, float) and not isinstance(
            salt_concentration, int
        ):
            raise TypeError("salt_concentration must be of type int or float")
        self.salt_concentration = salt_concentration

        if not isinstance(pH, float) and not isinstance(pH, int):
            raise TypeError("pH must be of type int or float")
        self.pH = pH

        if not isinstance(protein_concentration, float) and not isinstance(
            protein_concentration, int
        ):
            raise TypeError("protein_concentration must be of type int or float")
        self.protein_concentration = protein_concentration

        # Follow this scheme
        # {"V": V, "to": well}
        self.transfers = transfers

        return None

    def __repr__(self):
        return (
            str({"name": self.name}) + "\n" + str({"transfers": self.transfers}) + "\n"
        )

In [3]:
def load_protocol_and_labware(sim=True, api_level="2.11"):
    """
    Parameters
    ----------
    sim : bool
        If sim is false, this function will connect to the robot. Default: False.
    api_level: str
        The api level used for the protocol. Defaults to 2.11.
    """

    global protocol, tips_20, tips_300, reservoir, plate, eppi_rack,concical_rack, pip20, pip300

    if sim:
        protocol = simulate.get_protocol_api(api_level)
    else:
        protocol = execute.get_protocol_api(api_level)
        protocol.home()

    tips_300 = [protocol.load_labware("opentrons_96_tiprack_300ul", 9)]
    tips_300[0].set_offset(x=-0.30, y=0.00, z=-0.30)

    # reservoir = protocol.load_labware("nest_12_reservoir_15ml", 2)

    tips_20 = [
        protocol.load_labware("opentrons_96_filtertiprack_20ul", 8),
    ]
    tips_20[0].set_offset(x=-0.10, y=0.60, z=0.50)

    plate = protocol.load_labware("corning_384_wellplate_112ul_flat", 4)
    plate.set_offset(x=-0.40, y=1.00, z=0.00)

    eppi_rack = protocol.load_labware(
        "opentrons_24_tuberack_eppendorf_1.5ml_safelock_snapcap", 5
    )
    eppi_rack.set_offset(x=-0.40, y=1.40, z=0.30) 
    
    concical_rack = protocol.load_labware(
        "opentrons_10_tuberack_nest_4x50ml_6x15ml_conical", 6
    )
    concical_rack.set_offset(x=0, y=0, z=0.0)

    pip20 = protocol.load_instrument("p20_single_gen2", "right", tip_racks=tips_20)
    pip300 = protocol.load_instrument("p300_single_gen2", "left", tip_racks=tips_300)

    pip20.starting_tip = tips_20[0].well("A1")
    pip300.starting_tip = tips_300[0].well("A1")

## Start the protocol here:

In [4]:
load_protocol_and_labware(sim=True)

/data/robot_settings.json not found. Loading defaults
/data/deck_calibration.json not found. Loading defaults


## Change variables to your gusto here:

In [5]:
#cs_protein = [0, 
              # 25e-9, 
              #51e-9,  
              #100e-9, 
              #200e-9,
              #400e-9,
              #600e-9,
              #800e-9,
              #1000e-9]
#cs_salt = [101e-6,
           #150e-6,
           #200e-6,
           #300e-6, 
           #500e-6]
#V_total = 100

In [7]:
cs_protein = [0,
              #30e-9,
              51e-9,
              #100e-9,
              #200e-9,
              #400e-9,
              #600e-9,
              800e-9,
              1000e-9]
cs_salt = [101e-6,
           #150e-6,
           200e-6,
           #300e-6,
           #400e-6,
           500e-6]
V_total = 100

SyntaxError: invalid syntax (<ipython-input-7-f2401ed97750>, line 10)

Create fluid variables:

In [ ]:
buffer_without_salt = Fluid("Buffer without salt", concical_rack["A1"], 0, 9, [])
buffer_with_salt = Fluid("Buffer with salt", eppi_rack["A1"], 1000e-06, 9, [])
protein = Fluid("Protein", eppi_rack["D1"], 500e-06, 9, [], 5e-6)

In [ ]:
offsetY = int(input("Enter the well plate offset in Y (default: 2): ") or "2")
offsetX = int(input("Enter the well plate offset in X (default: 3): ") or "3")

This create a list of the wells that will be used, called used_wells.

In [ ]:
shape = (len(cs_salt), len(cs_protein))
shape_names = ("salt_concentration", "protein_concentration")

wells = np.array(plate.wells()).reshape(24, 16).swapaxes(0, 1)

used_wells = wells[offsetY : shape[0] + offsetY, offsetX : shape[1] + offsetX]

print("These wells will be used:")
for r in used_wells:
    row = "\t"
    for w in r:
        row += str(w)[:3] + "\t"
    print(row)

### Calculations

In [ ]:
protein.transfers = []
buffer_with_salt.transfers = []
buffer_without_salt.transfers = []

V_protein_total = 0

for r_index, c_salt in enumerate(cs_salt):
    for c_index, c_protein in enumerate(cs_protein):

        V_protein = V_total * c_protein / protein.protein_concentration
        V_protein = round(V_protein, 2)
        V_protein_total += V_protein
        
        # Raise error if volume is smaller than the minimum pipetting volume of the robot
        assert (
            V_protein >= 0.9999 or V_protein == 0
        ), "V_protein needs to be larger than 1 µl. Decrease stock concentration or increase total well volume"

        V_salt_buffer = V_total - V_protein
        

        n_salt_in_protein = V_protein * protein.salt_concentration
        n_salt = V_total * c_salt
        n_salt_to_add = n_salt - n_salt_in_protein

        V_buffer_with_salt = n_salt_to_add / buffer_with_salt.salt_concentration
        V_buffer_without_salt = V_salt_buffer - V_buffer_with_salt
        V_buffer_with_salt = round(V_buffer_with_salt, 2)
        V_buffer_without_salt = round(V_buffer_without_salt, 2)

        well_to_pipette_to = used_wells[r_index][c_index]

        protein.transfers.append({"V": V_protein, "to": well_to_pipette_to})

        buffer_without_salt.transfers.append({"V": V_buffer_without_salt, "to": well_to_pipette_to})
        
        buffer_with_salt.transfers.append({"V": V_buffer_with_salt, "to": well_to_pipette_to})
        

        # Raise error if salt concentration cannot be reached due to high protein volumes / high salt concentration in stock.
        assert n_salt_to_add > 0, f"Salt concentration of {c_salt} cannot be reached"
        
print(f"You will need AT LEAST {V_protein_total:.2f}µl of protein!")

# Robot Functions:

1. Pipette buffers

In [ ]:
t_start = datetime.datetime.now()
pip = pip300
for b in [buffer_without_salt, buffer_with_salt]:
    if not pip.hw_pipette["has_tip"]:
        pip.pick_up_tip()
    for t in b.transfers:

        pip.transfer(t["V"], b.location, t["to"], new_tip="never", touch_tip=True)

        setattr(t["to"], b.name.replace(" ", "_"), t["V"])

    pip.drop_tip()

In [ ]:
t_start = datetime.datetime.now()
pip = pip20
for b in [buffer_without_salt, buffer_with_salt]:
    if not pip.hw_pipette["has_tip"]:
        pip.pick_up_tip()
    for t in b.transfers:

        pip.transfer(t["V"], b.location, t["to"], new_tip="never", touch_tip=True)

        setattr(t["to"], b.name.replace(" ", "_"), t["V"])

    pip.drop_tip()

## --start of new adjustment--

In [ ]:
# pipetting buffer without salt
t_start = datetime.datetime.now()
pip = pip300
for b in [buffer_without_salt]:
    if not pip.hw_pipette["has_tip"]:
        pip.pick_up_tip()
    for t in b.transfers:

        pip.transfer(t["V"], b.location, t["to"], new_tip="never", touch_tip=True)

        setattr(t["to"], b.name.replace(" ", "_"), t["V"])

    pip.drop_tip()

In [ ]:
#pipetting buffer with salt
t_start = datetime.datetime.now()
pip = pip20
for b in [buffer_with_salt]:
    if not pip.hw_pipette["has_tip"]:
        pip.pick_up_tip()
    for t in b.transfers:

        pip.transfer(t["V"], b.location, t["to"], new_tip="never", touch_tip=True)

        setattr(t["to"], b.name.replace(" ", "_"), t["V"])

    pip.drop_tip()

## --end of new adjustments--

In [ ]:
for l in protocol.commands():
    print(l)
protocol.clear_commands()

2. Pipette protein

If you want to change the order of pipetting, e.g. from bottom-right to top-left instead of vice versa

In [ ]:
# used_wells_rearranged = np.flip(used_wells, axis=1)
# print(used_wells_rearranged)

In [ ]:
t_start = datetime.datetime.now()
pip = pip20
    
for t in protein.transfers:
    if not pip.hw_pipette["has_tip"]:
        pip.pick_up_tip()
    pip.transfer(t["V"], protein.location, t["to"], new_tip="never", touch_tip=True)

    setattr(t["to"], "protein", t["V"])

    pip.drop_tip()
t_end = datetime.datetime.now()

In [ ]:
#pip.drop_tip()
protocol.home()

# Save experimental details to file

In [ ]:
wells = {}
for w in used_wells.flatten():

    # print(w.well_name)
    wells[w.well_name] = {}

    for b in [buffer_with_salt, buffer_without_salt]:
        if hasattr(w, b.name.replace(" ", "_")):
            # print(f'\t{getattr(w,b.name.replace(" ","_"))} of {b.name}')
            wells[w.well_name][b.name] = getattr(w, b.name.replace(" ", "_"))

    # print(f"\t{w.protein:.1f} of protein")
    wells[w.well_name]["Protein"] = w.protein

In [ ]:
wells["Buffer with salt"] = buffer_with_salt.__dict__ 
wells["Buffer without salt"] = buffer_without_salt.__dict__ 
wells["Protein"] = protein.__dict__

wells["Start time"] = str(t_start)
wells["End time"] = str(t_end)

output_file_name = "experimental_details.toml"
with open(output_file_name, "w", encoding="UTF-16") as f:
    toml.dump(wells, f)

# This is just for experiment design to understand. Does not work on the robot

In [ ]:
import ipywidgets as widgets

from IPython.display import display

In [ ]:
salt_slider = widgets.FloatSlider(
    value=500,
    min=0,
    max=1000,
    step=1,
    description='Minimum salt concentration in µmol:',
    layout = widgets.Layout(griwidth='100%')
 
)

protein_slider = widgets.FloatSlider(
    value=500,
    min=0,
    max=1000,
    step=1,
    description='Max protein concentration in nmol:',
    layout = widgets.Layout(griwidth='100%')

)

protein_salt_slider = widgets.FloatSlider(
    value=500,
    min=0,
    max=1000,
    step=1,
    description='Salt concentration in protein stock in µmol:',
    layout = widgets.Layout(griwidth='100%')
)

In [ ]:
min_protein_concentration = protein.protein_concentration / V_total
print(f'The minum protein concentration is {1e9*min_protein_concentration:.2f}nm')

In [ ]:
def f(c_salt_min, c_protein_max, c_salt_in_protein):
    c_salt_min *= 1e-6
    c_protein_max *= 1e-9
    c_salt_in_protein *= 1e-6
    
    V_protein = V_total * c_protein_max / protein.protein_concentration
    V_protein = round(V_protein, 2)

    V_salt_buffer = V_total - V_protein

    n_salt_in_protein = V_protein * c_salt_in_protein
    n_salt = V_total * c_salt_min
    n_salt_to_add = n_salt - n_salt_in_protein

    if n_salt_to_add < 0 or V_protein < 1:
        return "Does not work, try again you honk"
    
    if n_salt_to_add >= 0 and V_protein >= 1:
        return "You did it, cowboy!"

   

In [ ]:
widgets.interact(f, c_salt_min = salt_slider, c_protein_max = protein_slider, c_salt_in_protein = protein_salt_slider);